# 03 — Depth & Severity Analysis

Investigate the correlation between MiDaS-estimated depth and the composite
severity score produced by `SeverityScorer`.

In [ ]:
import sys; sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.detection.severity_scorer import SeverityScorer
from src.detection.types import AnomalyDetection, BoundingBox

scorer = SeverityScorer()

In [ ]:
# Sweep depth and area, observe severity response
rows = []
for depth in range(0, 260, 20):
    for size in (40, 120, 260, 400):
        det = AnomalyDetection(
            class_id=0, class_name='pothole', confidence=0.85,
            bbox=BoundingBox(0, 0, size, size), area_px=size * size, depth_mm=depth,
        )
        scorer.score_one(det, 640, 480)
        rows.append({'depth_mm': depth, 'box': size, 'severity': det.severity_score, 'level': str(det.severity_level)})
df = pd.DataFrame(rows)
df.head()

In [ ]:
for size, grp in df.groupby('box'):
    plt.plot(grp['depth_mm'], grp['severity'], marker='o', label=f'box={size}px')
plt.xlabel('Depth (mm)'); plt.ylabel('Severity score'); plt.legend(); plt.title('Depth vs severity'); plt.show()

In [ ]:
# Severity-level distribution across the sweep
df['level'].value_counts().reindex(['LOW','MEDIUM','HIGH','CRITICAL']).plot.bar(title='Severity level counts'); plt.show()